# Análisis Exploratorio de Datos (EDA) - OSINT Medio Oriente

Este notebook cumple con el requisito del taller de realizar un EDA completo para entender las distribuciones, relaciones, problemas de calidad y posibles sesgos en los datos multifuente recopilados.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Carga del Dataset Integrado
Cargaremos el dataset diario reconstruido con fuentes abiertas reales. En la versión actual la base principal es Wikimedia Pageviews, actividad de revisiones de Wikipedia y titulares RSS/Google News; GDELT, ACLED y NASA FIRMS quedan como fuentes opcionales cuando entregan datos reales.

In [ ]:
base_dir = os.path.dirname(os.getcwd())
data_path = os.path.join(base_dir, 'data', 'processed', 'dataset_diario.csv')

df = pd.read_csv(data_path)
df['date'] = pd.to_datetime(df['date'])
df.head()

## 2. Calidad de Datos (Valores Nulos y Cobertura)
Un problema típico en OSINT es el _reporting lag_ o datos faltantes en tiempo real.

In [ ]:
missing_data = df.isnull().sum()
print("Datos Faltantes por Columna:")
print(missing_data[missing_data > 0])

# Verificar que todas las fechas estén cubiertas
print(f"\nRango de fechas: {df['date'].min().date()} a {df['date'].max().date()}")
print(f"Total de días: {len(df)}")

### Hallazgo de Calidad:
Al unificar fuentes de distintas frecuencias, los campos opcionales sin observaciones reales se completan en cero para preservar la granularidad diaria. Esto no implica imputación sintética de eventos: solo distingue entre señales observadas y fuentes no disponibles.

## 3. Distribución de la Señal de Presión Pública
El `escalation_score` conserva el nombre histórico del proyecto, pero en esta formulación representa un índice de presión pública observado a partir de atención Wikimedia/Wikipedia. El target supervisado es `target_high_escalation_next_day`: si el día siguiente cae en el cuartil alto de esa presión.

In [ ]:
sns.histplot(df['media_pressure_score'], bins=50, kde=True)
plt.title('Distribución del Índice de Presión Pública')
plt.xlabel('Índice (0-100)')
plt.ylabel('Frecuencia')
plt.show()

## 4. Correlación de Señales OSINT Disponibles
Analizamos relaciones lineales entre la presión pública, el volumen de atención Wikimedia, revisiones de Wikipedia, titulares RSS/Google News y fuentes opcionales como GDELT/FIRMS cuando existan datos reales.

In [ ]:
candidate_features = [
    'media_pressure_score',
    'wiki_pageviews_total',
    'wiki_revision_count',
    'volume',
    'avg_tone',
    'rss_total',
    'firms_hotspots_count',
]
features = [col for col in candidate_features if col in df.columns and df[col].nunique() > 1]
corr_matrix = df[features].corr()

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matriz de Correlación de Señales OSINT Observadas')
plt.show()

### Hallazgos Principales:
1. **Wikimedia/Wikipedia:** La señal central mide atención pública y actividad editorial, no eventos violentos verificados. Por eso el resultado debe interpretarse como presión pública observable.
2. **Fuente textual RSS/Google News:** Los titulares recientes se integran como conteos diarios y como archivo crudo trazable con título, descripción, fuente y enlace.
3. **Sesgo de cobertura:** Wikipedia, GDELT y RSS tienden a reflejar atención en inglés y medios dominantes, por lo que pueden sobrerrepresentar ciertos encuadres del conflicto.